In [0]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import java.text.SimpleDateFormat
import java.util.Date
import java.time.{LocalDate, ZoneId}
import java.time.format.DateTimeFormatter

// Inicializar SparkSession
val spark = SparkSession.builder()
  .appName("IngestLandingToDeltaCheckinRespuestas")
  .getOrCreate()

var landingPathBase = dbutils.widgets.get("landingPathBase")
var deltaPath = dbutils.widgets.get("deltaPath")
//var landingPathBase = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/canales/atenciones_suc/tch_encuestas_checkin_respuestas/landing/"
//var deltaPath = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/canales/atenciones_suc/tch_encuestas_checkin_respuestas/raw"
// Definir esquema explícito para el CSV
val schema = StructType(Seq(
  StructField("ID_TURNO", StringType, nullable = true),
  StructField("ID_PREGUNTA", IntegerType, nullable = true),
  StructField("VALOR_RESPUESTA", StringType, nullable = true),
  StructField("VERBATIM_RESPUESTA", StringType, nullable = true),
  StructField("FECHA_HORA_RESPUESTA", TimestampType, nullable = true),
  StructField("filename_spark", StringType, nullable = true),
  StructField("ts", StringType, nullable = true)
))

// Definir widget para el parámetro de mes (formato yyyy-MM)
dbutils.widgets.text("processMonth", "", "Month to process (yyyy-MM, leave empty for current month)")

// Obtener el mes a procesar
val processMonth = dbutils.widgets.get("processMonth")
//val processMonth = "2025-03"

val (year, month) = if (processMonth.isEmpty) {
  val currentDate = LocalDate.now(ZoneId.of("UTC-04:00"))
  (currentDate.format(DateTimeFormatter.ofPattern("yyyy")), 
   currentDate.format(DateTimeFormatter.ofPattern("MM")))
} else {
  val parts = processMonth.split("-")
  if (parts.length != 2) throw new IllegalArgumentException("processMonth debe tener formato yyyy-MM")
  (parts(0), parts(1))
}

// Mostrar el período que se está procesando
println(s"Procesando el período: $year-$month")

// Construir la ruta de lectura para el mes especificado o actual
val landingPath = s"$landingPathBase/$year/$month"

// Leer datos de landing con esquema explícito
val dfLanding = spark.read
  .option("delimiter", ";")
  .option("header", "true")
  .schema(schema)
  .csv(s"$landingPath/*")

println(s"Datos en landing ($landingPath):")
dfLanding.show(5)
val landingCount = dfLanding.count()
println(s"Total de registros en landing: $landingCount")

// Preparar datos: agregar bigdata_close_date y bigdata_ctrl_id, omitir filename_spark y ts
val currentTs = new Date().getTime
val formatter = new SimpleDateFormat("yyyyMMddHHmmss")
val bigdataCtrlId = s"${formatter.format(currentTs)}001".toLong

val dfProcessed = dfLanding
  .filter(col("ID_TURNO").isNotNull && col("ID_PREGUNTA").isNotNull && col("FECHA_HORA_RESPUESTA").isNotNull)
  .withColumnRenamed("ID_TURNO", "id_turno")
  .withColumnRenamed("ID_PREGUNTA", "id_pregunta")
  .withColumnRenamed("VALOR_RESPUESTA", "valor_respuesta")
  .withColumnRenamed("VERBATIM_RESPUESTA", "verbatim_respuesta")
  .withColumnRenamed("FECHA_HORA_RESPUESTA", "fecha_hora_respuesta")
  .withColumn("bigdata_close_date", to_date(col("fecha_hora_respuesta"), "yyyy-MM-dd HH:mm:ss"))
  .withColumn("bigdata_ctrl_id", lit(bigdataCtrlId).cast("bigint"))
  .drop("filename_spark", "ts")
  .dropDuplicates()

println("Datos procesados (sin nulos en id_turno, id_pregunta, fecha_hora_respuesta):")
dfProcessed.show(5)

// Verificar si la tabla Delta existe usando dbutils.fs
val deltaTableExists = dbutils.fs.ls(deltaPath).nonEmpty

// Leer datos existentes en Delta solo si existe, sino usar DataFrame vacío
val (dfDelta, deltaCount) = if (deltaTableExists) {
  val df = spark.read.format("delta").load(deltaPath)
  (df, df.count())
} else {
  println(s"La tabla Delta en $deltaPath no existe. Se creará con esta ejecución.")
  val emptyDf = spark.createDataFrame(new java.util.ArrayList[org.apache.spark.sql.Row](), dfProcessed.schema)
  (emptyDf, 0L)
}

println(s"Total de registros en Delta antes de la ingesta: $deltaCount")

// Identificar datos nuevos usando clave compuesta
val uniqueKey = Seq("id_turno", "id_pregunta", "fecha_hora_respuesta")
val dfNewData = if (deltaTableExists) {
  dfProcessed.join(
    dfDelta.select(uniqueKey.map(col): _*),
    uniqueKey,
    "left_anti"
  )
} else {
  dfProcessed
}

println("Datos nuevos a ingestar:")
dfNewData.show(5)
val newDataCount = dfNewData.count()
println(s"Total de registros nuevos a ingestar: $newDataCount")

if (newDataCount == 0) {
  println("No hay datos nuevos para ingestar.")
  dbutils.notebook.exit("No new data")
}

// Preparar particiones y escribir en Delta
val partitionCols = Seq("year", "month", "day")
val dfFinal = dfNewData
  .withColumn("year", expr("cast(extract(year from bigdata_close_date) as string)"))
  .withColumn("month", expr("lpad(cast(extract(month from bigdata_close_date) as string), 2, '0')"))
  .withColumn("day", expr("lpad(cast(extract(day from bigdata_close_date) as string), 2, '0')"))

// Escribir en Delta con modo "overwrite" si es la primera vez, "append" si ya existe
val writeMode = if (deltaTableExists) "append" else "overwrite"

dfFinal.write
  .format("delta")
  .partitionBy(partitionCols: _*)
  .mode(writeMode)
  .save(deltaPath)

// Contar registros finales de manera segura
val finalDeltaCount = try {
  spark.read.format("delta").load(deltaPath).count()
} catch {
  case e: Exception =>
    println(s"No se pudo contar los registros finales debido a: ${e.getMessage}. Usando conteo aproximado.")
    deltaCount + newDataCount // Aproximación basada en datos previos + nuevos
}

println(s"Total de registros en Delta después de la ingesta: $finalDeltaCount")
println("Ingesta completada con éxito.")
dbutils.notebook.exit("Success")